In [ ]:
# import torch
# print("torch file:", torch.__file__)
# print("torch version:", torch.__version__)
# print("has _utils:", hasattr(torch, "_utils"))
# %pip install -U --force-reinstall torch
# %pip install -U torch

In [ ]:
import os
import re
import sys
import random
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:

# --- Repo + pip (Colab: NEVER reinstall torch from requirements.txt — it replaces
#     the image's cu128 torch and breaks cuda-toolkit / dynamo / _utils, etc.) ---
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass
if not IN_COLAB and os.path.isdir("/content"):
    IN_COLAB = True


def _pip_install_requirements(req: Path, *, skip_torch: bool) -> None:
    if not req.is_file():
        return
    if skip_torch:
        kept = []
        for line in req.read_text().splitlines():
            s = line.split("#", 1)[0].strip()
            if not s:
                continue
            if re.match(r"(?i)^torch\b", s):
                continue
            kept.append(line)
        tmp = Path("/tmp/stk_mat2011_requirements_no_torch.txt")
        tmp.write_text("\n".join(kept) + "\n")
        target = tmp
        print("Colab: pip installing requirements (torch line skipped — using VM torch)")
    else:
        target = req
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(target)], check=False)


REPO_URL = "https://github.com/egil10/stk-mat2011.git"
if IN_COLAB:
    REPO_DIR = Path("/content/stk-mat2011")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
    _pip_install_requirements(REPO_DIR / "requirements.txt", skip_torch=True)
    _nb = REPO_DIR / "code" / "notebooks"
    if _nb.is_dir():
        os.chdir(str(_nb))
    print("Working directory:", os.getcwd())
else:
    REPO_DIR = Path.cwd().resolve().parents[1]
    _pip_install_requirements(REPO_DIR / "requirements.txt", skip_torch=False)

import torch
from torch import optim

SCRIPTS_DIR = (REPO_DIR / "code" / "scripts").resolve()
_nn = SCRIPTS_DIR / "nnhmm.py"
if not _nn.is_file():
    raise FileNotFoundError(
        f"nnhmm.py not found at {_nn}.\n"
        f"REPO_DIR={REPO_DIR.resolve()}\n"
        "On Colab: git clone must include code/scripts/nnhmm.py (push from local if missing on GitHub)."
    )

# Load by file path — avoids sys.path / name shadowing issues on Colab
import importlib.util
_spec = importlib.util.spec_from_file_location("nnhmm", str(_nn))
_nnhmm_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_nnhmm_mod)
NNHMM = _nnhmm_mod.NNHMM

# Optional: also put scripts on path for hmmlearn helpers etc.
_sd = str(SCRIPTS_DIR)
if _sd not in sys.path:
    sys.path.insert(0, _sd)

print("NNHMM loaded from:", _nn)


In [ ]:
SEED = 42
K = 2

# ---- master-file discovery ----
SYMBOL = "eurusd"  # file prefix used by p_duka/hw1-style names

# ---- model scale controls ----
MODEL_MAX_POINTS = 200_000  # if your series is huge, we auto-stride down to this
MODEL_STRIDE_MANUAL = None  # set an int to force stride; if None, auto
CLIP_Q = (0.001, 0.999)  # set to None to disable clipping
DROP_ZERO_RETURNS = True

# ---- training controls ----
EPOCHS = 400
PATIENCE = 30
LR = 1e-2

# ---- forecasting / evaluation ----
ALPHA = 0.05     # 95% intervals
Z95 = 1.959963984540054  # norm.ppf(0.975)

PLOT_N = 4000   # how many last test points to plot

# ---- reproducibility ----
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def interval_score_2sided(y, lo, hi, alpha=0.05):
    """
    Two-sided interval score for (1-alpha) prediction interval [lo, hi].
    Lower is better.
    """
    y = np.asarray(y)
    lo = np.asarray(lo)
    hi = np.asarray(hi)
    width = hi - lo
    below = (y < lo)
    above = (y > hi)
    penalty_below = (lo - y) * below
    penalty_above = (y - hi) * above
    return width + (2.0 / alpha) * (penalty_below + penalty_above)

In [ ]:
# Find/load a hw1 master parquet automatically (most recent match)
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)
    OUT_BASES = [
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed"),
        Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data"),
    ]
else:
    OUT_BASES = [Path("../data/processed_mega"), Path("../data/processed")]

patterns = [
    f"*{SYMBOL}*master*.parquet",
    f"*mega_master*.parquet",
    "*master_returns*.parquet",
    "*master*.parquet",
]

candidates = []
for base in OUT_BASES:
    if not base.exists():
        continue
    for pat in patterns:
        for p in base.glob(pat):
            # hw1 writes *_daily_counts.parquet next to the real master; same glob can match both
            if "daily_counts" in p.name:
                continue
            candidates.append((p.stat().st_mtime, p))

if not candidates:
    raise FileNotFoundError("No master parquet found. Put correct hw1 output path in this notebook.")

_, data_path = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
print("Using master file:", data_path)

df = pd.read_parquet(data_path)

# Expected columns: datetime + returns
if "returns" not in df.columns:
    raise ValueError(f"'returns' column not found in {data_path}. Columns: {list(df.columns)[:20]}")

# Make sure datetime exists
if "datetime" not in df.columns:
    raise ValueError(f"'datetime' column not found in {data_path}. Columns: {list(df.columns)[:20]}")

df = df.dropna(subset=["returns", "datetime"]).copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

# Clean returns
if DROP_ZERO_RETURNS:
    df = df[df["returns"] != 0].copy()

if CLIP_Q is not None:
    lo, hi = df["returns"].quantile(CLIP_Q[0]), df["returns"].quantile(CLIP_Q[1])
    df["returns"] = df["returns"].clip(lo, hi)

# Subsample for speed
if MODEL_STRIDE_MANUAL is not None:
    stride = int(MODEL_STRIDE_MANUAL)
else:
    stride = 1
    if len(df) > MODEL_MAX_POINTS:
        stride = int(np.ceil(len(df) / MODEL_MAX_POINTS))

print(f"Original points: {len(df):,} | Using stride={stride}")
if stride > 1:
    df = df.iloc[::stride].copy().reset_index(drop=True)

print("Final points:", len(df))
display(df.head())
display(df.tail())

In [ ]:
# Time-based split
n = len(df)
i1 = int(0.70 * n)
i2 = int(0.85 * n)

df_tr = df.iloc[:i1].copy()
df_va = df.iloc[i1:i2].copy()
df_te = df.iloc[i2:].copy()

print("train:", len(df_tr), "val:", len(df_va), "test:", len(df_te))
print("train range:", df_tr["datetime"].min(), "->", df_tr["datetime"].max())
print("test range:", df_te["datetime"].min(), "->", df_te["datetime"].max())

In [ ]:
def to_tensor(x):
    return torch.tensor(x.values, dtype=torch.float32).unsqueeze(1).to(device)

y_tr = to_tensor(df_tr["returns"])
y_va = to_tensor(df_va["returns"])
y_te = to_tensor(df_te["returns"])

print(y_tr.shape, y_va.shape, y_te.shape)

In [ ]:
model = NNHMM(n_states=K, input_dim=1).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)

best_val = np.inf
best_state = None
bad = 0

hist_tr, hist_va, hist_lr = [], [], []

In [ ]:
N_PRINTS = 20
LOG_STEP = max(1, EPOCHS // N_PRINTS)
print(f"{'ep':>6} | {'train_nll':>12} | {'val_nll':>12} | {'lr':>10}")
print("-" * 52)

for ep in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    ll_tr = model(y_tr)                 # total log-likelihood
    loss_tr = -ll_tr / len(y_tr)      # per-sample NLL
    loss_tr.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        ll_va = model(y_va)
        loss_va = -ll_va / len(y_va)

    scheduler.step(loss_va)

    tr = float(loss_tr.item())
    va = float(loss_va.item())
    lr = optimizer.param_groups[0]["lr"]

    hist_tr.append(tr)
    hist_va.append(va)
    hist_lr.append(lr)

    if va < best_val - 1e-6:
        best_val = va
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1

    if ep % LOG_STEP == 0:
        print(f"{ep:6d} | {tr:12.6f} | {va:12.6f} | {lr:10.2e}")

    if bad >= PATIENCE:
        print(f"{ep:6d} | {tr:12.6f} | {va:12.6f} | {lr:10.2e}  (early stop)")
        print(f"early stop at ep={ep}, best_val={best_val:.6f}")
        break

model.load_state_dict(best_state)
print("best val NLL per sample:", best_val)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(hist_tr, label="train_nll")
plt.plot(hist_va, label="val_nll")
plt.title("NNHMM training")
plt.xlabel("epoch")
plt.ylabel("NLL per sample")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10,3))
plt.plot(hist_lr)
plt.title("learning rate")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
with torch.no_grad():
    trans = model.transition_matrix.detach().cpu().numpy()
    pi = model.pi.detach().cpu().numpy()
    mu = model.means.detach().cpu().numpy().ravel()
    sig = torch.exp(model.log_std).detach().cpu().numpy().ravel()

print("pi:", pi)
print("transition:\n", trans)
for k in range(K):
    print(f"state {k}: mean={mu[k]:.6e} std={sig[k]:.6e}")

In [ ]:
# Viterbi state shading on the last part of the test set
PLOT_START = max(0, len(y_te) - PLOT_N)
y_plot = y_te[PLOT_START:].contiguous()

with torch.no_grad():
    z_plot = model.predict_states(y_plot, method="viterbi").detach().cpu().numpy()

x_plot = df_te.iloc[PLOT_START:]["datetime"].values
y_plot_np = df_te.iloc[PLOT_START:]["returns"].values

plt.figure(figsize=(14,4))
plt.plot(x_plot, y_plot_np, color="black", lw=0.6, label="returns")

for k in range(K):
    mask = (z_plot == k)
    plt.scatter(x_plot[mask], y_plot_np[mask], s=8, alpha=0.9, label=f"state {k}")

plt.title(f"NNHMM Viterbi states (last {len(x_plot)} test points)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# AR(1) baseline on train, evaluate on test (1-step ahead)
r_tr = df_tr["returns"].values
r_va = df_va["returns"].values
r_te = df_te["returns"].values

# Fit on train: r_t = a + b r_{t-1} + e_t
X = np.column_stack([np.ones(len(r_tr)-1), r_tr[:-1]])
y_ = r_tr[1:]
a_hat, b_hat = np.linalg.lstsq(X, y_, rcond=None)[0]

# One-step ahead predictions for test:
# prev for te[0] is last val point; prev for te[i] is te[i-1]
prev_te = np.concatenate([r_va[-1:], r_te[:-1]])
pred_ar1 = a_hat + b_hat * prev_te

true = r_te
rmse_ar1 = float(np.sqrt(np.mean((true - pred_ar1)**2)))

# Homoskedastic interval using residual std from train
res = y_ - (a_hat + b_hat * r_tr[:-1])
s_hat = float(res.std(ddof=1))

lo_ar1 = pred_ar1 - Z95 * s_hat
hi_ar1 = pred_ar1 + Z95 * s_hat
cov_ar1 = float(np.mean((true >= lo_ar1) & (true <= hi_ar1)))

score_ar1 = float(interval_score_2sided(true, lo_ar1, hi_ar1, alpha=ALPHA).mean())

print(f"AR(1) a={a_hat:.6e}, b={b_hat:.6e}, s={s_hat:.6e}")
print("AR(1) RMSE:", rmse_ar1)
print("AR(1) 95% coverage:", cov_ar1)
print("AR(1) interval score:", score_ar1)

In [ ]:
# NNHMM: NLL on test + prediction intervals (mixture + hard switch)
model.eval()
with torch.no_grad():
    # Test NLL per sample (as in your training objective)
    nll_te = float((-model(y_te) / len(y_te)).item())

    # Build full sequence for filtered state probs
    y_all = torch.cat([y_tr, y_va, y_te], dim=0)  # [T_all, 1]
    T_all = y_all.shape[0]

    N_tr = len(y_tr)
    N_va = len(y_va)
    test_start = N_tr + N_va
    N_te = len(y_te)

    # Forward log-alpha for all times
    log_alpha = model.forward_log_alpha(y_all)  # [T_all, K]

    # Filtered state probs at time t: p(z_t=k | x_1:t)
    log_norm = torch.logsumexp(log_alpha, dim=1, keepdim=True)
    filt = torch.exp(log_alpha - log_norm)  # [T_all, K]

    # Predict distribution of z_t given x_1:t-1 for all test times:
    # for test observation at index t, we need filt at t-1
    # test indices are t in [test_start, T_all-1]
    filt_slice = filt[test_start-1 : test_start-1 + N_te]  # [N_te, K]
    trans = model.transition_matrix.detach()                  # [K, K]
    mu = model.means.detach().view(K)                       # [K]
    sigma2 = torch.exp(model.log_std.detach()).view(K) ** 2  # [K]

    # p(z_t | x_1:t-1)
    state_pred = filt_slice @ trans                       # [N_te, K]

    # Mixture predictive mean/var via law of total variance
    mix_mean = state_pred @ mu                            # [N_te]
    exp_var = state_pred @ sigma2                        # E[Var(Y|Z)]
    exp_mean2 = state_pred @ (mu**2)                    # E[E[Y|Z]^2]
    mix_var = exp_var + exp_mean2 - mix_mean**2
    mix_std = torch.sqrt(torch.clamp(mix_var, min=1e-12))

    lo_mix = mix_mean - Z95 * mix_std
    hi_mix = mix_mean + Z95 * mix_std

    # Hard switch (argmax over predicted state prob at t)
    k_star = torch.argmax(state_pred, dim=1)             # [N_te]
    hard_mean = mu[k_star]                               # [N_te]
    hard_var = sigma2[k_star]                           # [N_te]
    hard_std = torch.sqrt(torch.clamp(hard_var, min=1e-12))

    lo_hard = hard_mean - Z95 * hard_std
    hi_hard = hard_mean + Z95 * hard_std

# Convert to numpy for scoring/coverage
true = df_te["returns"].values
mix_mean_np = mix_mean.cpu().numpy()
lo_mix_np = lo_mix.cpu().numpy()
hi_mix_np = hi_mix.cpu().numpy()

hard_mean_np = hard_mean.cpu().numpy()
lo_hard_np = lo_hard.cpu().numpy()
hi_hard_np = hi_hard.cpu().numpy()

rmse_mix = float(np.sqrt(np.mean((true - mix_mean_np)**2)))
cov_mix = float(np.mean((true >= lo_mix_np) & (true <= hi_mix_np)))
score_mix = float(interval_score_2sided(true, lo_mix_np, hi_mix_np, alpha=ALPHA).mean())

rmse_hard = float(np.sqrt(np.mean((true - hard_mean_np)**2)))
cov_hard = float(np.mean((true >= lo_hard_np) & (true <= hi_hard_np)))
score_hard = float(interval_score_2sided(true, lo_hard_np, hi_hard_np, alpha=ALPHA).mean())

print("NNHMM test NLL per sample:", nll_te)
print("NNHMM mixture RMSE:", rmse_mix)
print("NNHMM mixture 95% coverage:", cov_mix)
print("NNHMM mixture interval score:", score_mix)
print("NNHMM hard-switch RMSE:", rmse_hard)
print("NNHMM hard-switch 95% coverage:", cov_hard)
print("NNHMM hard-switch interval score:", score_hard)

In [ ]:
# Plot mixture interval vs true returns on the last part of test
N = min(PLOT_N, len(df_te))
x = df_te.iloc[-N:]["datetime"].values
y_true = df_te.iloc[-N:]["returns"].values

lo = lo_mix_np[-N:]
hi = hi_mix_np[-N:]
pred = mix_mean_np[-N:]

plt.figure(figsize=(14,4))
plt.plot(x, y_true, color="black", lw=0.7, label="true")
plt.plot(x, pred, color="blue", lw=1.0, alpha=0.9, label="mix mean")

plt.fill_between(x, lo, hi, color="dodgerblue", alpha=0.2, label="95% interval (mixture approx)")

plt.title("NNHMM mixture forecast interval (test tail)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
run = {
    "seed": SEED,
    "K": K,
    "n_train": len(df_tr),
    "n_val": len(df_va),
    "n_test": len(df_te),
    "val_best_nll_per_sample": float(best_val),
    "nn_hmm_test_nll_per_sample": float(nll_te),

    "ar1_rmse": float(rmse_ar1),
    "ar1_cov95": float(cov_ar1),
    "ar1_interval_score": float(score_ar1),

    "nn_mix_rmse": float(rmse_mix),
    "nn_mix_cov95": float(cov_mix),
    "nn_mix_interval_score": float(score_mix),

    "nn_hard_rmse": float(rmse_hard),
    "nn_hard_cov95": float(cov_hard),
    "nn_hard_interval_score": float(score_hard),
}

summary = pd.DataFrame([run])
display(summary)
summary